# DIS08 Project – Rental Prices in NRW
## Phase: Scrub

Goal: Clean the NRW subset of the dataset and create derived variables
(e.g., price per square meter) in a reproducible way.


In [1]:
import pandas as pd

file_path = "../data/raw/immo_data.csv"
df = pd.read_csv(file_path)

df_nrw = df[df["regio1"] == "Nordrhein_Westfalen"].copy()
df_nrw.shape


(62863, 49)

In [4]:
cols = [
    "regio2", "regio3",
    "baseRent", "totalRent",
    "livingSpace", "noRooms",
    "balcony", "hasKitchen", "newlyConst",
    "yearConstructed"
]

df_nrw_clean = df_nrw[cols].copy()
df_nrw_clean.shape, df_nrw_clean.isna().sum()


((62863, 10),
 regio2                 0
 regio3                 0
 baseRent               0
 totalRent          12104
 livingSpace            0
 noRooms                0
 balcony                0
 hasKitchen             0
 newlyConst             0
 yearConstructed    12052
 dtype: int64)

### Regeln

- baseRent > 0 → keine kostenlosen / fehlerhaften Einträge

- livingSpace > 0 → Quadratmeter nötig

- price_per_sqm berechnen

- Entfernen extremer Ausreißer (unter 1 €, über 50 €)

In [5]:
# Basisfilter: gültige Miete und Wohnfläche
df_nrw_clean = df_nrw_clean[
    (df_nrw_clean["baseRent"] > 0) &
    (df_nrw_clean["livingSpace"] > 0)
].copy()

# Preis pro Quadratmeter berechnen
df_nrw_clean["price_per_sqm"] = (
    df_nrw_clean["baseRent"] / df_nrw_clean["livingSpace"]
)

# Ausreißer entfernen
df_nrw_clean = df_nrw_clean[
    (df_nrw_clean["price_per_sqm"] >= 1) &
    (df_nrw_clean["price_per_sqm"] <= 50)
]

df_nrw_clean.shape, df_nrw_clean["price_per_sqm"].describe()


((62824, 11),
 count    62824.000000
 mean         8.046672
 std          3.194358
 min          1.000000
 25%          6.000000
 50%          7.192982
 75%          9.193548
 max         50.000000
 Name: price_per_sqm, dtype: float64)

In [6]:
# Save cleaned NRW dataset for later phases
df_nrw_clean.to_csv("../data/processed/nrw_clean.csv", index=False)


In [7]:
import os
os.path.exists("../data/processed/nrw_clean.csv")


True